In [ ]:
# Langchain imports


# API Keys


# import libraries

# **IMPORTING FEATURES**
<hr>


Import features from other notebooks using import_ipynb library

In [ ]:
# Import Features
import import_ipynb
import os
import pandas as pd
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
features = {
    "missing_vals": "Handles missing values in dataset",
    "summaries": "Generates dataset summary and statistics"
}
load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)
router_prompt = ChatPromptTemplate.from_template("""
You are a query routing agent for a data cleaning system.

You must decide which feature should handle the user request.

Available features:
{features}

User request:
{query}

Return ONLY the feature name.
""")

# **QUERY ROUTER**
<hr>

The query router uses an LLM to process your query and pass it into your defined features

In [ ]:
def route_query(user_query, df):
    print(f"ROUTE QUERY RECEIVED: {user_query}")

    feature_text = "\n".join([f"- {k}: {v}" for k, v in features.items()])

    prompt = router_prompt.invoke({
        "features": feature_text,
        "query": user_query
    })

    response = llm.invoke(prompt)
    selected_feature = response.content.strip()

    print(f"SELECTED FEATURE: {selected_feature}")

    if selected_feature == "missing_vals":
        from missing_vals import missing_vals
        return missing_vals(df, user_query)

    elif selected_feature == "summaries":
        from summaries import summaries
        return summaries(df, user_query)

    else:
        print("No valid feature found. Returning original df.")
        return df

# **TEST QUERIES**

In [ ]:
dataset_name = "sample-data.csv"

load_dotenv()
PROJECT_ROOT = os.environ["PROJECT_ROOT"]
path = f"{PROJECT_ROOT}\\datasets\\{dataset_name}"

df = pd.read_csv(path)
test_df = df.copy()

result = route_query("fill missing values using median", test_df)